In [17]:
import math
import random
from dataclasses import dataclass
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [7]:
# Load raw traffic file
traffic_raw = pd.read_csv("cleaned_traffic_data.csv")

# Basic cleaning
traffic_raw["Timestamp"] = pd.to_datetime(traffic_raw["Timestamp"])
traffic_raw["Station"] = traffic_raw["Station"].astype(str)

# Keep only the columns we need for now
traffic_core = traffic_raw[["Timestamp", "Station", "Route", "Total Flow", "Avg Speed"]].copy()

# Make sure numeric columns are numeric
traffic_core["Total Flow"] = pd.to_numeric(traffic_core["Total Flow"], errors="coerce")
traffic_core["Avg Speed"] = pd.to_numeric(traffic_core["Avg Speed"], errors="coerce")
traffic_core["Route"] = pd.to_numeric(traffic_core["Route"], errors="coerce")

print("traffic_core shape:", traffic_core.shape)
display(traffic_core.head())

traffic_core shape: (4114680, 5)


,Timestamp,Station,Route,Total Flow,Avg Speed
0,2024-10-01,308512,50,497.0,64.1
1,2024-10-01,311831,5,27.0,NaN
2,2024-10-01,311832,5,78.0,NaN
3,2024-10-01,311844,5,43.0,NaN
4,2024-10-01,311847,5,73.0,NaN


In [8]:
# Build wide flow table
flow_df = traffic_core.pivot_table(
    index="Timestamp",
    columns="Station",
    values="Total Flow",
    aggfunc="mean"
).sort_index()

# Build wide speed table
speed_df = traffic_core.pivot_table(
    index="Timestamp",
    columns="Station",
    values="Avg Speed",
    aggfunc="mean"
).sort_index()

print("flow_df shape:", flow_df.shape)
print("speed_df shape:", speed_df.shape)

display(flow_df.head())
display(speed_df.head())

flow_df shape: (2208, 1806)
speed_df shape: (2208, 1162)


Station,3001021,3001022,3001101,3001102,3001111,3001121,3003011,3003041,3003044,3003054,...,3423063,3423064,3423065,3423066,3423091,3423094,3900021,3900022,3900023,3900024
Timestamp,,,,,,,,,,,,,,,,,,,,,
2024-10-01 00:00:00,121.0,63.0,11.0,113.0,2.0,0.0,45.0,16.0,5.0,1.0,...,2.0,2632.0,2.0,NaN,75.0,0.0,556.0,320.0,546.0,320.0
2024-10-01 01:00:00,98.0,59.0,11.0,84.0,26.0,26.0,31.0,27.0,0.0,1.0,...,3.0,2824.0,1.0,NaN,47.0,20.0,520.0,225.0,505.0,225.0
2024-10-01 02:00:00,99.0,18.0,11.0,94.0,3.0,23.0,35.0,19.0,4.0,0.0,...,3.0,3184.0,4.0,NaN,43.0,27.0,482.0,201.0,519.0,201.0
2024-10-01 03:00:00,97.0,25.0,6.0,75.0,0.0,0.0,25.0,15.0,9.0,0.0,...,2.0,3263.0,1.0,NaN,61.0,15.0,522.0,246.0,583.0,246.0
2024-10-01 04:00:00,188.0,64.0,18.0,161.0,32.0,41.0,47.0,43.0,3.0,0.0,...,18.0,3654.0,6.0,NaN,153.0,79.0,649.0,446.0,915.0,446.0


Station,3001021,3001102,3001111,3001121,3004071,3005031,3005041,3005052,3007031,3007033,...,3423021,3423024,3423061,3423064,3423091,3423094,3900021,3900022,3900023,3900024
Timestamp,,,,,,,,,,,,,,,,,,,,,
2024-10-01 00:00:00,65.3,66.8,66.2,68.2,69.1,64.9,64.9,65.0,66.7,67.4,...,65.8,68.2,66.8,29.1,67.0,68.2,66.5,68.1,66.6,68.1
2024-10-01 01:00:00,65.6,65.1,61.8,61.8,68.5,65.0,64.8,65.1,66.1,67.2,...,65.8,68.2,57.9,32.9,65.7,63.1,66.8,67.8,66.0,67.8
2024-10-01 02:00:00,61.8,63.9,65.9,61.7,68.0,64.9,65.0,64.9,65.9,66.9,...,65.1,68.2,69.6,43.9,66.0,63.4,65.9,67.6,65.4,67.6
2024-10-01 03:00:00,65.1,66.4,68.3,68.2,67.4,64.9,64.9,64.9,66.3,66.5,...,65.6,68.2,66.7,56.4,65.7,61.5,66.2,67.7,66.0,67.7
2024-10-01 04:00:00,65.4,66.8,61.9,61.9,67.0,64.3,64.5,64.8,65.4,66.2,...,66.4,68.2,68.5,58.1,65.6,63.1,66.7,67.9,68.1,67.9


In [9]:
print("Same time index:", flow_df.index.equals(speed_df.index))
print("Same station columns:", flow_df.columns.equals(speed_df.columns))
print("Missing flow values:", flow_df.isna().sum().sum())
print("Missing speed values:", speed_df.isna().sum().sum())

Same time index: True
Same station columns: False
Missing flow values: 176744
Missing speed values: 33610


In [10]:
station_meta_tmp = (
    traffic_core[["Station", "Route"]]
    .drop_duplicates()
    .rename(columns={"Station": "station_id", "Route": "route"})
    .copy()
)

station_meta_tmp["station_id"] = station_meta_tmp["station_id"].astype(str)

print("station_meta_tmp shape:", station_meta_tmp.shape)
display(station_meta_tmp.head())

station_meta_tmp shape: (1896, 2)


,station_id,route
0,308512,50
1,311831,5
2,311832,5
3,311844,5
4,311847,5


In [11]:
xls = pd.ExcelFile("pems_output.xlsx")
print(xls.sheet_names)

['Report Data', 'PeMS Report Description']


In [12]:
meta_raw = pd.read_excel("pems_output.xlsx", sheet_name=0)

print("meta_raw shape:", meta_raw.shape)
print(meta_raw.columns.tolist())
display(meta_raw.head())

meta_raw shape: (1861, 15)
['Fwy', 'District', 'County', 'City', 'CA PM', 'Abs PM', 'Length', 'ID', 'Name', 'Lanes', 'Type', 'Sensor Type', 'HOV', 'MS ID', 'IRM']


,Fwy,District,County,City,CA PM,Abs PM,Length,ID,Name,Lanes,Type,Sensor Type,HOV,MS ID,IRM
0,I5-N,3,Sacramento,NaN,1.919,497.212,4.312,3413014,5NB at Twin Cities Rd,2,Mainline,NaN,No,1,NaN
1,I5-N,3,Sacramento,NaN,2.026,497.319,NaN,3413016,5NB to Twin Cities Rd,4,Off Ramp,NaN,No,1,NaN
2,I5-N,3,Sacramento,NaN,9.498,504.791,3.291,317802,Hood Franklin Rd,2,Mainline,radars,No,1,NaN
3,I5-N,3,Sacramento,NaN,10.942,506.235,3.115,3013091,5NB at Elk Grove HOV,1,HOV,NaN,24H,1,NaN
4,I5-N,3,Sacramento,NaN,11.08,506.373,NaN,311844,Elk Grove Blvd 5NB Slip,2,On Ramp,others,No,1,NaN


In [13]:
# Find common stations
common_stations = flow_df.columns.intersection(speed_df.columns)

print("Common stations:", len(common_stations))

# Filter both
flow_df = flow_df[common_stations].copy()
speed_df = speed_df[common_stations].copy()

# Final check
print("Now same columns:", flow_df.columns.equals(speed_df.columns))
print("New shape:", flow_df.shape)

Common stations: 1162
Now same columns: True
New shape: (2208, 1162)


In [14]:
station_meta = meta_raw[["ID", "Fwy", "Abs PM"]].copy()

station_meta = station_meta.rename(columns={
    "ID": "station_id",
    "Fwy": "route",
    "Abs PM": "abs_pm"
})

station_meta["station_id"] = station_meta["station_id"].astype(str)

# Keep only stations we use
station_meta = station_meta[station_meta["station_id"].isin(flow_df.columns)]

print("station_meta shape:", station_meta.shape)
display(station_meta.head())

station_meta shape: (1154, 3)


,station_id,route,abs_pm
0,3413014,I5-N,497.212
2,317802,I5-N,504.791
3,3013091,I5-N,506.235
5,312134,I5-N,506.373
9,3013103,I5-N,507.464


In [15]:
print("Flow shape:", flow_df.shape)
print("Speed shape:", speed_df.shape)
print("Meta shape:", station_meta.shape)

print("Stations match:",
      set(flow_df.columns).issubset(set(station_meta["station_id"])))

Flow shape: (2208, 1162)
Speed shape: (2208, 1162)
Meta shape: (1154, 3)
Stations match: False


# Traffic Forecasting Robustness Study

This notebook builds a leakage-aware traffic forecasting pipeline using PeMS data and compares:

- Random Forest
- LSTM
- CNN-GRU-LSTM
- GraphWaveNet
- GraphWaveNet-GRU
- GraphWaveNet-LSTM
- GraphWaveNet-GRU-LSTM

The study evaluates:
1. Clean forecasting performance
2. Sensor outage robustness
3. Temporal missingness robustness
4. Noise injection robustness

The main goal is to test whether progressive recurrent refinement improves GraphWaveNet, especially at longer horizons and under degraded sensing conditions.

## Configuration

This section defines the forecasting setup:
- 24-hour input window
- 12h, 24h, 48h, 72h prediction horizons
- training parameters
- model size

In [18]:
@dataclass
class Config:
    seed: int = 42
    input_len: int = 24
    horizons: Tuple[int, ...] = (12, 24, 48, 72)
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    batch_size: int = 32
    max_epochs: int = 50
    patience: int = 8
    lr: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 5.0
    hidden_dim: int = 64
    gwn_blocks: int = 3
    tcn_kernel: int = 2
    dropout: float = 0.2
    rf_n_estimators: int = 300
    rf_max_depth: Optional[int] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

CFG = Config()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

print(CFG)
print("Device:", CFG.device)

Config(seed=42, input_len=24, horizons=(12, 24, 48, 72), train_ratio=0.7, val_ratio=0.15, batch_size=32, max_epochs=50, patience=8, lr=0.001, weight_decay=1e-05, grad_clip=5.0, hidden_dim=64, gwn_blocks=3, tcn_kernel=2, dropout=0.2, rf_n_estimators=300, rf_max_depth=None, device='cuda')
Device: cuda
